# 03 — SVG Construction (multi-city)

Builds one Street View Graph (`HeteroData`) per point, using `02`'s cached
segmentation output -- looped over every point in `01`'s combined,
multi-city `reconciled_points.parquet` (city-prefixed `point_id`, no
city-specific logic needed here): island-labels the fused stuff classes
(Building, Vegetation, Crosswalk, Lane Marking), pairs them with native
thing instances (Signage, Light_pole classes), computes Ego's scene-level
features, constructs `sees`/`mounted_with`/`near` edges, and saves the
graph plus a QC overlay image -- both checkpointed, per point.

**Revised:** `mounted_with` (see `src/svg_builder.py` /
`configs/svg_schema.yaml`) is now triggered by centroid distance ALONE --
bbox overlap is no longer required to connect two pole-family objects
(a sign and a light mounted on the same pole rarely have overlapping
boxes at all).

Uses `src/features.py`, `src/svg_builder.py`, `src/svg_visualize.py`,
`src/manifest.py`, `src/segmentation.py` (for `load_result`/palette only).
CPU is sufficient -- no GPU needed for this notebook.

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# torch is expected pre-installed on Colab. torch_geometric's basic
# Data/HeteroData objects don't need the compiled scatter/sparse
# extensions — those are only required later for actual GNN layers (06/07).
!pip install -q torch_geometric scipy pillow tqdm pyyaml pandas numpy matplotlib

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/svg_schema.yaml") as f:
    svg_cfg = yaml.safe_load(f)

INTERIM_DIR = Path(paths_cfg["interim_dir"])
PROCESSED_DIR = Path(paths_cfg["processed_dir"])

SEG_DIR = INTERIM_DIR / "segmentation"
SVG_OUT_DIR = PROCESSED_DIR / "svg_graphs"
VIZ_OUT_DIR = INTERIM_DIR / "svg_visualizations"
LOG_PATH = INTERIM_DIR / "svg_construction_log.csv"

SVG_OUT_DIR.mkdir(parents=True, exist_ok=True)
VIZ_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Provisional parameters in effect this run (confirm/override in svg_schema.yaml):")
for k in ["near_cutoff_d", "mask_min_area_fraction", "mounted_with_distance_threshold"]:
    print(f"  {k}: {svg_cfg[k]}")

In [ ]:
import manifest
import segmentation as seg
import features
import svg_builder
import svg_visualize

import torch
print(f"torch: {torch.__version__}")
import torch_geometric
print(f"torch_geometric: {torch_geometric.__version__}")

In [ ]:
# ── Load reconciled points from 01 ──────────────────────────────────────
import pandas as pd

reconciled = pd.read_parquet(INTERIM_DIR / "reconciled_points.parquet")
all_point_ids = reconciled["point_id"].tolist()
path_lookup = dict(zip(reconciled["point_id"], reconciled["image_path"]))
# Used below instead of point_id.startswith("positive_"/"negative_") --
# that pattern no longer matches now that point_id is city-prefixed
# (e.g. "bog_positive_123"). Looking the class up from the reconciled
# dataframe is robust regardless of id format.
class_lookup = dict(zip(reconciled["point_id"], reconciled["class"]))

pending = manifest.pending_items(all_point_ids, SVG_OUT_DIR, ext=".pt")
print(f"Total points: {len(all_point_ids)}  |  Already done: {len(all_point_ids) - len(pending)}  |  Pending: {len(pending)}")

# Sanity check: every pending point should already have a segmentation
# result from 02 — flag any that don't, rather than failing deep inside the loop.
missing_seg = [pid for pid in pending if not manifest.is_done(SEG_DIR, pid, ext=".npz")]
if missing_seg:
    print(f"⚠️  {len(missing_seg)} pending points have NO segmentation output from 02 yet — "
          f"these will fail below. Run 02 first for: {missing_seg[:10]}{'...' if len(missing_seg) > 10 else ''}")

In [ ]:
# ── Main per-point loop — checkpointed, resumable ───────────────────────
from PIL import Image
from tqdm.auto import tqdm

for point_id in tqdm(pending, desc="Building SVG"):
    try:
        seg_map, segments_info = seg.load_result(SEG_DIR, point_id)
        image = Image.open(path_lookup[point_id]).convert("RGB")
        w, h = image.size
        assert seg_map.shape == (h, w), (
            f"Segmentation shape {seg_map.shape} doesn't match image size {(h, w)} "
            f"for {point_id} — no resizing should ever occur between 02 and 03."
        )

        svf = features.compute_svf(seg_map, segments_info)
        enclosure = features.compute_enclosure(
            seg_map, segments_info, svg_cfg["enclosure_classes"],
            crop_fraction=svg_cfg["enclosure_crop_fraction_from_top"],
        )
        entropy = features.compute_entropy(seg_map, segments_info)

        objects = svg_builder.extract_objects(
            seg_map, segments_info,
            svg_cfg["mask_min_area_fraction"],
            svg_cfg.get("mask_min_area_fraction_by_type", {}),
        )

        data = svg_builder.assemble_svg(
            objects, w, h, svf, enclosure, entropy,
            near_cutoff_d=svg_cfg["near_cutoff_d"],
            mounted_with_threshold=svg_cfg["mounted_with_distance_threshold"],
        )

        torch.save(data, SVG_OUT_DIR / f"{point_id}.pt")

        fig = svg_visualize.render_svg_overlay(image, seg_map, segments_info, objects, data, w, h)
        svg_visualize.save_overlay(fig, VIZ_OUT_DIR, point_id)

        manifest.append_log(LOG_PATH, point_id, "svg_construction", "ok")

    except Exception as e:
        manifest.append_log(LOG_PATH, point_id, "svg_construction", "error", str(e))
        tqdm.write(f"  ⚠️  {point_id}: {e}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
log = manifest.load_log(LOG_PATH)
n_remaining = len(manifest.pending_items(all_point_ids, SVG_OUT_DIR, ext=".pt"))
print(f"Remaining pending after this run: {n_remaining} / {len(all_point_ids)}")

if "status" in log.columns and (log["status"] == "error").any():
    errors = log[log["status"] == "error"]
    print(f"\n⚠️  {len(errors)} points failed — re-running this notebook will retry them.")
    display(errors[["point_id", "error", "timestamp"]].tail(20))
else:
    print("\n✅ No errors logged.")

In [ ]:
# ── QC: node-count distribution across the dataset — worth checking that
#    graphs aren't degenerating to near-empty or absurdly large. ─────────
import seaborn as sns
import matplotlib.pyplot as plt

node_counts = []
sample_ids = [pid for pid in all_point_ids if manifest.is_done(SVG_OUT_DIR, pid, ext=".pt")]
for pid in tqdm(sample_ids, desc="Scanning graph sizes"):
    data = torch.load(SVG_OUT_DIR / f"{pid}.pt", weights_only=False)
    n_objects = sum(data[nt].x.shape[0] for nt in ["signage", "light_pole", "road_marking", "building", "vegetation"])
    node_counts.append({"point_id": pid, "n_objects": n_objects})

counts_df = pd.DataFrame(node_counts)
print(counts_df["n_objects"].describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(counts_df["n_objects"], bins=30, ax=ax)
ax.set_xlabel("Detected objects per point (excluding Ego)")
plt.tight_layout()
plt.savefig(INTERIM_DIR / "qc_svg_node_count_distribution.png", dpi=150)
plt.show()

n_zero = (counts_df["n_objects"] == 0).sum()
if n_zero:
    print(f"\n⚠️  {n_zero} points have ZERO detected objects (Ego-only graph) — "
          f"expected occasionally, but worth spot-checking their visualizations "
          f"in {VIZ_OUT_DIR} if this is a large fraction of the dataset.")

In [ ]:
point_ids = counts_df.loc[counts_df["n_objects"] == 0, "point_id"]

print(point_ids)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set Seaborn theme
sns.set_theme(style="whitegrid", palette="pastel")

fig, ax = plt.subplots(figsize=(4, 4))

sns.histplot(
    data=counts_df,
    x="n_objects",
    bins=30,
    kde=True,
    color="grey",
    ax=ax
)

ax.set_xlabel("Detected objects per point (excluding Ego)", size=10)
ax.set_ylabel("Count", size=10)

plt.tight_layout()
plt.savefig(INTERIM_DIR / "qc_svg_node_count_distribution.png", dpi=150)
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sns.set_theme(style="whitegrid", palette="pastel")

# mean = solid, median = long dash, Q1/Q3 = dash-dot
LINE_STYLES = {
    "mean":   dict(ls="-", lw=2.6),
    "median": dict(ls=(0, (3, 3)), lw=1.8),
    "quart":  dict(ls=(0, (3, 2, 1, 2)), lw=1.8),
}

fig, ax = plt.subplots(figsize=(5, 4))

sns.histplot(
    data=counts_df,
    x="n_objects",
    bins=30,
    kde=True,
    color="grey",
    alpha=0.55,
    ax=ax,
)

# --- stat lines -------------------------------------------------------
vals = counts_df["n_objects"].to_numpy()
mean = vals.mean()
q1, q2, q3 = np.percentile(vals, [25, 50, 75])

for val, key in [(mean, "mean"), (q2, "median"), (q1, "quart"), (q3, "quart")]:
    ax.axvline(val, color="black", zorder=6, **LINE_STYLES[key])

ax.text(
    0.97, 0.98,
    f"mean = {mean:.1f}\nQ1 = {q1:.0f}\nmedian = {q2:.0f}\nQ3 = {q3:.0f}",
    transform=ax.transAxes, ha="right", va="top", size=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", lw=0.1, alpha=0.9),
    zorder=7,
)
# ----------------------------------------------------------------------

ax.set_title(f"all (n={len(counts_df)})", size=11)
ax.set_xlabel("Detected objects per point (excluding Ego)", size=10)
ax.set_ylabel("Count", size=10)

ax.legend(
    handles=[
        Line2D([], [], color="black", label="mean", **LINE_STYLES["mean"]),
        Line2D([], [], color="black", label="median (Q2)", **LINE_STYLES["median"]),
        Line2D([], [], color="black", label="Q1 / Q3", **LINE_STYLES["quart"]),
    ],
    loc="upper right", bbox_to_anchor=(0.97, 0.72),
    frameon=False, framealpha=0.9, fontsize=8,
)

plt.tight_layout()
plt.savefig(INTERIM_DIR / "qc_svg_node_count_distribution_ALL_legend.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sns.set_theme(style="whitegrid", palette="pastel")

# mean = solid, median = long dash, Q1/Q3 = dash-dot
LINE_STYLES = {
    "mean":   dict(ls="-", lw=2.6),
    "median": dict(ls=(0, (3, 3)), lw=1.8),
    "quart":  dict(ls=(0, (3, 2, 1, 2)), lw=1.8),
}

fig, ax = plt.subplots(figsize=(5, 4))

sns.histplot(
    data=counts_df,
    x="n_objects",
    bins=30,
    kde=True,
    color="grey",
    alpha=0.55,
    ax=ax,
)

# --- stat lines -------------------------------------------------------
vals = counts_df["n_objects"].to_numpy()
mean = vals.mean()
q1, q2, q3 = np.percentile(vals, [25, 50, 75])

for val, key in [(mean, "mean"), (q2, "median"), (q1, "quart"), (q3, "quart")]:
    ax.axvline(val, color="black", zorder=6, **LINE_STYLES[key])

ax.text(
    0.97, 0.98,
    f"mean = {mean:.1f}\nQ1 = {q1:.0f}\nmedian = {q2:.0f}\nQ3 = {q3:.0f}",
    transform=ax.transAxes, ha="right", va="top", size=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", lw=0.1, alpha=0.9),
    zorder=7,
)
# ----------------------------------------------------------------------

ax.set_title(f"all (n={len(counts_df)})", size=11)
ax.set_xlabel("Detected objects per point (excluding Ego)", size=10)
ax.set_ylabel("Count", size=10)

# ax.legend(
#     handles=[
#         Line2D([], [], color="black", label="mean", **LINE_STYLES["mean"]),
#         Line2D([], [], color="black", label="median (Q2)", **LINE_STYLES["median"]),
#         Line2D([], [], color="black", label="Q1 / Q3", **LINE_STYLES["quart"]),
#     ],
#     loc="upper right", bbox_to_anchor=(0.97, 0.72),
#     frameon=False, framealpha=0.9, fontsize=8,
# )

plt.tight_layout()
plt.savefig(INTERIM_DIR / "qc_svg_node_count_distribution_ALL_nolegend.png", dpi=300)
plt.show()

In [ ]:
# ── QC: node-count distributions (positive / negative only) ─────────────
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="pastel")

NODE_TYPES = ["signage", "light_pole", "road_marking", "building", "vegetation"]

def scan_node_counts(point_ids, desc="Scanning graph sizes"):
    counts = []
    for pid in tqdm(point_ids, desc=desc):
        data = torch.load(SVG_OUT_DIR / f"{pid}.pt", weights_only=False)
        n_objects = sum(data[nt].x.shape[0] for nt in NODE_TYPES)
        counts.append({"point_id": pid, "n_objects": n_objects})
    return pd.DataFrame(counts)


def plot_node_count_distribution(counts_df, label, color, out_path):
    print(f"\n=== {label} (n={len(counts_df)}) ===")
    print(counts_df["n_objects"].describe())

    fig, ax = plt.subplots(figsize=(4, 4))
    sns.histplot(
        data=counts_df,
        x="n_objects",
        bins=30,
        kde=True,
        color=color,
        ax=ax,
    )
    ax.set_xlabel("Detected objects per point (excluding Ego)", size=10)
    ax.set_ylabel("Count", size=10)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()

    n_zero = (counts_df["n_objects"] == 0).sum()
    if n_zero:
        print(f"\n⚠️  [{label}] {n_zero} points have ZERO detected objects (Ego-only graph) — "
              f"expected occasionally, but worth spot-checking their visualizations "
              f"in {VIZ_OUT_DIR} if this is a large fraction of the dataset.")


# All done points, split by prefix
sample_ids = [pid for pid in all_point_ids if manifest.is_done(SVG_OUT_DIR, pid, ext=".pt")]
positive_ids = [pid for pid in sample_ids if class_lookup.get(pid) == "positive"]
negative_ids = [pid for pid in sample_ids if class_lookup.get(pid) == "negative"]

print(f"Total: {len(sample_ids)} | Positive: {len(positive_ids)} | Negative: {len(negative_ids)}")

# --- Positive only (red) ---
counts_df_pos = scan_node_counts(positive_ids, desc="Scanning graph sizes (positive)")
plot_node_count_distribution(
    counts_df_pos, "positive", "red", INTERIM_DIR / "qc_svg_node_count_distribution_positive.png"
)

# --- Negative only (blue) ---
counts_df_neg = scan_node_counts(negative_ids, desc="Scanning graph sizes (negative)")
plot_node_count_distribution(
    counts_df_neg, "negative", "blue", INTERIM_DIR / "qc_svg_node_count_distribution_negative.png"
)

In [ ]:
# ── QC: node-count distributions (positive / negative, single figure) ───
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sns.set_theme(style="whitegrid", palette="pastel")

NODE_TYPES = ["signage", "light_pole", "road_marking", "building", "vegetation"]

# line styles: mean = solid, median = long dash, Q1/Q3 = dash-dot
LINE_STYLES = {
    "mean":   dict(ls="-",  lw=2.6),
    "median": dict(ls=(0, (3, 3)), lw=1.8),
    "quart":  dict(ls=(0, (3, 2, 1, 2)), lw=1.8),
}


def scan_node_counts(point_ids, desc="Scanning graph sizes"):
    counts = []
    for pid in tqdm(point_ids, desc=desc):
        data = torch.load(SVG_OUT_DIR / f"{pid}.pt", weights_only=False)
        n_objects = sum(data[nt].x.shape[0] for nt in NODE_TYPES)
        counts.append({"point_id": pid, "n_objects": n_objects})
    return pd.DataFrame(counts)


def annotate_stats(ax, values):
    """Black vertical lines: mean (solid), median (dashed), Q1/Q3 (dash-dot)."""
    mean = values.mean()
    q1, q2, q3 = np.percentile(values, [25, 50, 75])

    for val, key in [(mean, "mean"), (q2, "median"), (q1, "quart"), (q3, "quart")]:
        ax.axvline(val, color="black", zorder=6, **LINE_STYLES[key])

    ax.text(
        0.97, 0.98,
        f"mean = {mean:.1f}\nQ1 = {q1:.0f}\nmedian = {q2:.0f}\nQ3 = {q3:.0f}",
        transform=ax.transAxes, ha="right", va="top", size=9,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", lw=0.1, alpha=0.9),
        zorder=7,
    )
    return mean, q1, q2, q3


# ── Collect ids ─────────────────────────────────────────────────────────
sample_ids = [pid for pid in all_point_ids if manifest.is_done(SVG_OUT_DIR, pid, ext=".pt")]
positive_ids = [pid for pid in sample_ids if class_lookup.get(pid) == "positive"]
negative_ids = [pid for pid in sample_ids if class_lookup.get(pid) == "negative"]

print(f"Total: {len(sample_ids)} | Positive: {len(positive_ids)} | Negative: {len(negative_ids)}")

counts_df_pos = scan_node_counts(positive_ids, desc="Scanning graph sizes (positive)")
counts_df_neg = scan_node_counts(negative_ids, desc="Scanning graph sizes (negative)")

# ── Shared bins (shared y comes from sharey=True) ───────────────────────
STAT = "count"   # "density" or "probability" if the two class sizes differ a lot
all_vals = pd.concat([counts_df_pos["n_objects"], counts_df_neg["n_objects"]])
bin_edges = np.histogram_bin_edges(all_vals, bins=30)

# ── Plot ────────────────────────────────────────────────────────────────
panels = [
    (counts_df_pos, "positive", "red"),
    (counts_df_neg, "negative", "blue"),
]

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)

for ax, (df, label, color) in zip(axes, panels):
    print(f"\n=== {label} (n={len(df)}) ===")
    print(df["n_objects"].describe())

    sns.histplot(
        data=df, x="n_objects", bins=bin_edges,
        stat=STAT, kde=True, color=color, alpha=0.55, ax=ax,
    )
    annotate_stats(ax, df["n_objects"].to_numpy())

    ax.set_title(f"{label} (n={len(df)})", size=11)
    ax.set_xlabel("Detected objects per point (excluding Ego)", size=10)
    ax.set_xlim(bin_edges[0], bin_edges[-1])

    n_zero = (df["n_objects"] == 0).sum()
    if n_zero:
        print(f"\n⚠️  [{label}] {n_zero} points have ZERO detected objects (Ego-only graph) — "
              f"expected occasionally, but worth spot-checking their visualizations "
              f"in {VIZ_OUT_DIR} if this is a large fraction of the dataset.")

axes[0].set_ylabel(STAT.capitalize(), size=10)
axes[1].set_ylabel("")

fig.legend(
    handles=[
        Line2D([], [], color="black", label="mean", **LINE_STYLES["mean"]),
        Line2D([], [], color="black", label="median (Q2)", **LINE_STYLES["median"]),
        Line2D([], [], color="black", label="Q1 / Q3", **LINE_STYLES["quart"]),
    ],
    loc="lower center", ncol=3, frameon=False, fontsize=9,
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(INTERIM_DIR / "qc_svg_node_count_distribution_PN.png", dpi=300)
plt.show()

In [ ]:
# ── QC: node-count distribution — ALL / positive / negative (2×2 grid) ──
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sns.set_theme(style="ticks")
# Verdana, with DejaVu Sans as fallback where Verdana isn't installed (Linux/Colab)
plt.rcParams["font.sans-serif"] = ["Verdana", "DejaVu Sans"]

NODE_TYPES = ["signage", "light_pole", "road_marking", "building", "vegetation"]

# mean = solid, median = long dash, Q1/Q3 = dash-dot
LINE_STYLES = {
    "mean":   dict(ls="-", lw=2.6),
    "median": dict(ls=(0, (3, 3)), lw=1.8),
    "quart":  dict(ls=(0, (3, 2, 1, 2)), lw=1.8),
}


def scan_node_counts(point_ids, desc="Scanning graph sizes"):
    counts = []
    for pid in tqdm(point_ids, desc=desc):
        data = torch.load(SVG_OUT_DIR / f"{pid}.pt", weights_only=False)
        n_objects = sum(data[nt].x.shape[0] for nt in NODE_TYPES)
        counts.append({"point_id": pid, "n_objects": n_objects})
    return pd.DataFrame(counts)


def annotate_stats(ax, values):
    """Black vertical lines: mean (solid), median (dashed), Q1/Q3 (dash-dot)."""
    mean = values.mean()
    q1, q2, q3 = np.percentile(values, [25, 50, 75])

    for val, key in [(mean, "mean"), (q2, "median"), (q1, "quart"), (q3, "quart")]:
        ax.axvline(val, color="black", zorder=6, **LINE_STYLES[key])

    ax.text(
        0.97, 0.98,
        f"mean = {mean:.1f}\nQ1 = {q1:.0f}\nmedian = {q2:.0f}\nQ3 = {q3:.0f}",
        transform=ax.transAxes, ha="right", va="top", size=11,  # size 8: Verdana runs wide
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", lw=0.1, alpha=0.9),
        zorder=7,
    )
    return mean, q1, q2, q3


# ── Collect ids and scan ────────────────────────────────────────────────
sample_ids = [pid for pid in all_point_ids if manifest.is_done(SVG_OUT_DIR, pid, ext=".pt")]
positive_ids = [pid for pid in sample_ids if class_lookup.get(pid) == "positive"]
negative_ids = [pid for pid in sample_ids if class_lookup.get(pid) == "negative"]

print(f"Total: {len(sample_ids)} | Positive: {len(positive_ids)} | Negative: {len(negative_ids)}")

counts_df_pos = scan_node_counts(positive_ids, desc="Scanning graph sizes (positive)")
counts_df_neg = scan_node_counts(negative_ids, desc="Scanning graph sizes (negative)")
counts_df = pd.concat([counts_df_pos, counts_df_neg], ignore_index=True)

for df, label in [(counts_df, "all"), (counts_df_pos, "positive"), (counts_df_neg, "negative")]:
    print(f"\n=== {label} (n={len(df)}) ===")
    print(df["n_objects"].describe())

    n_zero = (df["n_objects"] == 0).sum()
    if n_zero:
        print(f"\n⚠️  [{label}] {n_zero} points have ZERO detected objects (Ego-only graph) — "
              f"expected occasionally, but worth spot-checking their visualizations "
              f"in {VIZ_OUT_DIR} if this is a large fraction of the dataset.")

# ── Bins: "all" gets its own; positive/negative share a common set ──────
bins_all = np.histogram_bin_edges(counts_df["n_objects"], bins=30)
bins_pn = np.histogram_bin_edges(
    pd.concat([counts_df_pos["n_objects"], counts_df_neg["n_objects"]]), bins=30
)

# Panel position (row, col) -> (dataframe, label, color, bin edges)
PANELS = {
    (0, 0): (counts_df,     "all",      "grey", bins_all),
    (1, 0): (counts_df_pos, "positive", "red",  bins_pn),
    (1, 1): (counts_df_neg, "negative", "blue", bins_pn),
}

# No figure-level sharing: "all" keeps independent axes,
# positive/negative are linked to each other further below.
fig, axes = plt.subplots(2, 2, figsize=(9, 8))

for (r, c), (df, label, color, edges) in PANELS.items():
    ax = axes[r, c]
    sns.histplot(
        data=df, x="n_objects", bins=edges,
        kde=True, color=color, alpha=0.55, ax=ax,
    )
    annotate_stats(ax, df["n_objects"].to_numpy())

    ax.set_title(f"{label.upper()}  (n={len(df)})", size=11, loc="left")
    ax.set_xlim(edges[0], edges[-1])
    # top-left panel drops the x-label: the bottom row already carries it
    ax.set_xlabel("" if (r, c) == (0, 0) else "Detected objects per point (excluding Ego)", size=10)
    ax.set_ylabel("Count" if c == 0 else "", size=10)
    sns.despine(ax=ax)

# Link positive and negative so their bar heights stay comparable
ax_pos, ax_neg = axes[1, 0], axes[1, 1]
ax_pos.sharex(ax_neg)
ax_pos.sharey(ax_neg)
ax_pos.set_ylim(0, max(ax_pos.get_ylim()[1], ax_neg.get_ylim()[1]) * 1.05)
ax_neg.tick_params(labelleft=False)  # inner tick labels are redundant once shared

# ── Top-right cell: legend only ─────────────────────────────────────────
ax_legend = axes[0, 1]
ax_legend.set_axis_off()
ax_legend.legend(
    handles=[
        Line2D([], [], color="black", label="mean", **LINE_STYLES["mean"]),
        Line2D([], [], color="black", label="median (Q2)", **LINE_STYLES["median"]),
        Line2D([], [], color="black", label="Q1 / Q3", **LINE_STYLES["quart"]),
    ],
    loc="center", frameon=False, fontsize=11,
    handlelength=3.2, labelspacing=1.2,
)

plt.tight_layout()
plt.savefig(INTERIM_DIR / "qc_svg_node_count_distribution_grid.png", dpi=400)
plt.show()

In [ ]:
print(f"QC overlay images saved to: {VIZ_OUT_DIR}")
print(f"Legend: {REPO_DIR}/docs/svg_visualization_legend.md")
print()
print("Next: 04_tvg_construction.ipynb")